# Gafaelfawr token API

## Import libraries and symbols

In [ ]:
import os
from datetime import UTC, datetime
from urllib.parse import urljoin

from httpx import AsyncClient
from lsst.rsp import RSPDiscovery, RSPInternalDiscovery

## Obtain token

This checks that the user has a token.

In [ ]:
token = RSPDiscovery.get_token()
assert token, "You have no notebook token"

## Find the Gafaelfawr API

Currently, service discovery is done manually using the raw Repertoire API because internal services are not exposed to the service discovery built into notebooks. This probably should change in the future to make it easier to test internal services.

The client created here is not safe to use in general because it will unconditionally send the user's authentication to any site. Here, it is only used to talk to internal services, so we know that is safe.

In [ ]:
discovery = RSPInternalDiscovery()
session = discovery.get_session()
gafaelfawr_url = discovery.get_internal_service_url("gafaelfawr", version="v1")
print(f"Gafaelfawr API URL: {gafaelfawr_url}")

## Get user information

Get the user's user information from their notebook token.

In [ ]:
r = session.get(gafaelfawr_url + "/user-info")
assert r.status_code == 200
user_info = r.json()

## Print out username information

In [ ]:
print("Your username is", user_info["username"])

## Print out name and email

Name and email are optional and may not be set for every user (if, for instance, the RSP uses GitHub authentication and the user doesn't release an email address or name).

In [ ]:
if "name" in user_info:
    print("Your name is", user_info["name"])
if "email" in user_info:
    print("Your email address is", user_info["email"])

## Print out UID and group information

At the IDF there should be a per-user group matching the username, and all other groups should start with `g_`. Results inside mobu will of course be different and depend on the configuration of the test user.

In [ ]:
print("Your numeric UID is", user_info["uid"])
print("Your numeric GID is", user_info["gid"])
print("Your groups are", ", ".join(f"{g['name']} ({g['id']})" for g in user_info["groups"]))

## Print out quota information

`muster-quota` is an internal service used for testing and should be ignored.

In [ ]:
if "quota" not in user_info:
    print("You have no quotas set")
else:
    quota = user_info["quota"]
    if "api" in quota:
        print("Your API quotas:")
        for service, amount in sorted(quota["api"].items()):
            print(f"  Service {service}: {amount} per minute")
    if "notebook" in quota:
        notebook = quota["notebook"]
        if not notebook["spawn"]:
            print("You may not create a notebook server")
        else:
            cpu = quota["notebook"]["cpu"]
            memory = quota["notebook"]["memory"]
            print(f"You may create a notebook server with up to {cpu} core equivalents and {memory}GiB of memory")
    if "tap" in quota:
        print("Your TAP quotas:")
        for service, rule in sorted(quota["tap"].items()):
            print(f"  Backend {service}: {rule['concurrent']} concurrent reqeusts")

## Get token metadata

Now, retrieve the metadata about the user's token specifically.

In [ ]:
r = session.get(gafaelfawr_url + "/token-info")
assert r.status_code == 200
token_info = r.json()

## Print basic token information

In [ ]:
print("Your username is", token_info["username"])
print("Your token identifier is", token_info["token"])
print("Your token type is", token_info["token_type"], "(will always be notebook when executing in Nublado)")
print("Your scopes are", ", ".join(token_info["scopes"]))

## Print out expiration information

In [ ]:
created = datetime.fromtimestamp(token_info["created"], tz=UTC)
expires = datetime.fromtimestamp(token_info["expires"], tz=UTC)
current = datetime.now(tz=UTC)
print("Your token was issued at:", created.isoformat(sep=" "))
print("Your token expires at:   ", expires.isoformat(sep=" "))
print("The time is currently:   ", current.isoformat(sep=" ", timespec="seconds"))

## Check token validity

In [ ]:
assert current >= created, "Your token was created after the current time?!"
assert current <= expires, f"Your token expired at {expires.isoformat(sep=' ')}"
print("Your token is VALID")